# Run 1 of 8: `ministral-3:8b`, pipeline 1 (judge writes its own SQL)

One run of the 1,000 ClaimDB test claims. Results: `MyDrive/sqljudge/results/pipeline1_A/ministral-8b/test-public/`.

**Before running:** Runtime → Change runtime type → **L4 GPU** → Save, then Connect. Run the cells top to bottom.
If the runtime stops, run all cells again: finished claims are skipped and the run resumes.

## 1. Settings

In [ ]:
import os
os.environ["JUDGE"] = "ministral-3:8b"
os.environ["NAME"] = "ministral-8b"
os.environ["PIPELINE"] = "A"
os.environ["BUILDER"] = ""
os.environ["BUILDER_NAME"] = ""
os.environ["CLAIMS"] = "subsets/test-public.jsonl"
os.environ["DRIVE"] = "/content/drive/MyDrive/sqljudge"
os.environ["RESULTS"] = "/content/drive/MyDrive/sqljudge/results"
os.environ["BIRD_DB_ROOT"] = "/content/bird/dev_databases"
os.environ["OLLAMA_CONTEXT_LENGTH"] = "16384"
os.environ["REPO"] = "/content/claimdb-specialist-sql"
print({k: os.environ[k] for k in ["JUDGE", "PIPELINE", "BUILDER", "RESULTS"]})

## 2. Mount Google Drive (click Allow)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(os.environ["RESULTS"], exist_ok=True)

## 3. Setup: GPU check, Ollama, code, BIRD databases (from the Drive copy), models

In [ ]:
%%bash
set -e
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
apt-get -qq install -y zstd pciutils > /dev/null
curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
nohup ollama serve > /content/ollama.log 2>&1 &
until curl -s localhost:11434/api/version > /dev/null; do sleep 1; done; ollama --version
if [ -d "$REPO" ]; then git -C "$REPO" pull -q; else git clone -q https://github.com/Kushagra-Agnihotri/claimdb-specialist-sql.git "$REPO"; fi
pip install -q -r "$REPO/Navtej/requirements.txt"
git -C "$REPO" log --oneline -1
mkdir -p "$DRIVE/bird" /content/bird
if [ ! -f "$DRIVE/bird/dev_databases.zip" ]; then
  curl -sL --fail -o /content/dev.zip https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip
  unzip -q -o /content/dev.zip -d /content/bird_raw
  cp "$(find /content/bird_raw -name dev_databases.zip | head -1)" "$DRIVE/bird/dev_databases.zip"
fi
[ -f "$BIRD_DB_ROOT/formula_1/formula_1.sqlite" ] || unzip -q -o "$DRIVE/bird/dev_databases.zip" -d /content/bird
rm -rf /content/bird/__MACOSX; echo "BIRD databases: $(ls "$BIRD_DB_ROOT" | wc -l)"
echo "pulling $JUDGE"; ollama pull "$JUDGE" > /dev/null 2>&1
[ -n "$BUILDER" ] && { echo "pulling $BUILDER"; ollama pull "$BUILDER" > /dev/null 2>&1; }
ollama list

## 4. The run (about 12 h on an L4)

In [ ]:
%%bash
cd "$REPO/Navtej"
./pipeline1_A/run_A.sh --model "$JUDGE" --name "$NAME" --claims "$CLAIMS" --db-root "$BIRD_DB_ROOT" --results-root "$RESULTS"

## 5. Score

In [ ]:
%%bash
cd "$REPO/Navtej"
./pipeline1_A/score_A.sh --name "$NAME" --claims "$CLAIMS" --results-root "$RESULTS"

## Progress (optional; run only when cell 4 is not running in this tab, or just check the file on Drive)

In [ ]:
%%bash
f="$RESULTS/pipeline1_A/ministral-8b/test-public/results.jsonl"; [ -f "$f" ] && echo "$(wc -l < "$f") / 1000 claims done" || echo "not started"